<a href="https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

**Lane 2 — Refresh / Content Opportunity Scoring**
**Author:** Farrukh Imran · **Date:** September 12, 2026

**Deployed paper:** https://farrukh776.github.io/flyrank-ai/

This notebook mirrors the deployed research paper section by section. Every number quoted in the paper
is produced by a cell below.

## 0. Setup

*Clone the repo, connect to the warehouse, rebuild the March 2026 feature frame and label.*

The warehouse is gated on Hugging Face. The token is read from a Colab Secret named `HF_TOKEN` —
never pasted into a cell, because this repo is public.

In [ ]:
import os, subprocess, sys

if "google.colab" in sys.modules and not os.path.exists("flyrank-ai"):
    subprocess.run(["git", "clone", "https://github.com/Farrukh776/flyrank-ai.git"], check=True)
if os.path.basename(os.getcwd()) != "flyrank-ai":
    os.chdir("flyrank-ai")

%pip -q install duckdb

import duckdb, pandas as pd, numpy as np, json
from google.colab import userdata

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"   # mid-panel month. The final month is left sealed and was never used in development.

print("Working directory:", os.getcwd())
print("Month under study:", MONTH)

Working directory: /content/flyrank-ai
Month under study: 2026-03


## 1. Question

*The research question and the decision it supports.*

**Research question.** Does a simple, explainable rule actually identify the pages most likely to be
declining — or does it only look like it does?

**The decision this supports.** A content editor with limited review hours picks pages off a ranked list
and reviews or rewrites them. With hundreds of thousands of pages in the portfolio, something has to decide
what goes on this week's list.

**Cost of a wrong call, both directions.**

- *False positive* — an editor spends hours reviewing a page that was not actually a priority.
- *False negative* — a genuinely declining page keeps losing traffic, unnoticed, for another month.

**Why precision@K, not accuracy.** What matters is whether the *top of the list* is right, not whether every
row in a 176k-row dataset is classified correctly. An editor never sees row 40,000. Precision@K measures
exactly the thing the decision depends on.

**Why this needs data rather than intuition.** The assumption under test ("older content is more likely to be
declining") is widely held and easy to encode as a rule. Section 3 tests whether it survives contact with the
data. It does not, in the shape the rule assumes — and that finding is the spine of this paper.

In [ ]:
# The unit of analysis: one row = one pseudonymous content item, for one client,
# summarised over one calendar month (March 2026).
print("Unit of analysis : one content item x one client x one month")
print("Decision supported: which pages an editor reviews first")
print("Metric            : precision@K (K = 20, 50) - top-of-list quality")
print("Label             : within-month decline proxy (defined in Section 3)")

Unit of analysis : one content item x one client x one month
Decision supported: which pages an editor reviews first
Metric            : precision@K (K = 20, 50) - top-of-list quality
Label             : within-month decline proxy (defined in Section 3)


## 2. Data

*Which release, which tables, date windows, what was excluded and why. Public-safe.*

**Release.** FlyRank ML Internship warehouse, Hugging Face-hosted, Parquet, partitioned by month
(`FlyRank/internship-warehouse`, gated — free approval).

**Tables used.**

| Table | Grain | Used for |
|---|---|---|
| `fact_content_daily_performance` | one row per content item × client × day | impressions, clicks, position, availability flags |
| `dim_content` | one row per content item | creation date, word count, search volume, competition |

**Window.** The `month=2026-03` partition only — a mid-panel month. The dataset's final month is the natural
outcome window of any past→future label, so it is treated as a sealed test month and was never touched during
development.

**Excluded, and why:**

- **Product-computed decision flags** (`health_score`, `priority_score`, `action_type`) — not shipped in this
  dataset. Treated as a benchmark to beat, never as a feature or a label.
- **`last_optimized_date`** — first considered as the staleness signal. A distribution check (Section 3)
  showed *every* non-null value in the March slice falls **after** March 31, 2026, because `dim_content` is a
  current snapshot rather than a point-in-time table. Using it would have leaked future optimisation activity
  into a March-only feature set. Dropped.
- **GA4 engagement fields where `ga4_data_available` is false** — only 4.2% of March rows have active GA4
  tracking. Without that filter, "not tracked yet" would be silently read as "zero engagement".

**Public-safety.** No client names, no URLs, no raw search queries appear anywhere. Client and content
identifiers are pseudonymous hashes, used only for joining and for grouping the validation split — never as
model features.

In [ ]:
# --- Scale of the raw slice, and the availability check that drove an exclusion ---
scale = con.sql(f"""
SELECT
  COUNT(*)                                                     AS daily_rows,
  COUNT(DISTINCT content_hash_id)                              AS content_items,
  MIN(report_date)                                             AS first_day,
  MAX(report_date)                                             AS last_day,
  ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1)
                                                               AS ga4_available_pct
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()
print(scale.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 daily_rows  content_items  first_day   last_day  ga4_available_pct
    9841378         331437 2026-03-01 2026-03-31                4.2


In [ ]:
# --- Grain check: is one row really (day x client x content item)? Empty result = grain holds. ---
dupes = con.sql(f"""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
GROUP BY 1,2,3
HAVING c > 1
LIMIT 5
""").df()
print("Duplicate-grain rows found:", len(dupes), "(0 = the stated grain holds)")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate-grain rows found: 0 (0 = the stated grain holds)


In [ ]:
# --- Build the monthly feature frame: aggregate daily facts, join static content metadata ---
features = con.sql(f"""
SELECT
  f.content_hash_id,
  f.client_hash_id,
  SUM(f.gsc_impressions)                                          AS impressions_month,
  SUM(f.gsc_clicks)                                               AS clicks_month,
  SUM(f.gsc_sum_position) / NULLIF(SUM(f.gsc_impressions), 0)     AS avg_position,
  DATE '2026-03-31' - ANY_VALUE(d.content_created_date)           AS content_age_days,
  ANY_VALUE(d.word_count)                                         AS word_count,
  ANY_VALUE(d.search_volume)                                      AS search_volume,
  ANY_VALUE(d.competition)                                        AS competition
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet') f
LEFT JOIN read_parquet('{REL}/dim_content.parquet') d
  ON f.content_hash_id = d.content_hash_id
GROUP BY f.content_hash_id, f.client_hash_id
""").df()

# avg_position is impressions-weighted (sum of daily position sums / sum of impressions),
# NOT a naive mean of daily averages - otherwise a 2-impression day would count as much as a 2000-impression day.
features["ctr"] = features["clicks_month"] / features["impressions_month"].replace(0, np.nan)

print("Content items in the March slice:", f"{len(features):,}")
features.head(3)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Content items in the March slice: 331,437


,content_hash_id,client_hash_id,impressions_month,clicks_month,avg_position,content_age_days,word_count,search_volume,competition,ctr
0,content_b7e512995f79d5a6,client_73cda7b4e4f265ea,1140.0,2.0,4.450877,396,<NA>,20,0.00,0.001754
1,content_a7da352b73b02668,client_73cda7b4e4f265ea,4944.0,13.0,7.435680,396,2330,10,0.00,0.002629
2,content_d056587ff7faca0c,client_73cda7b4e4f265ea,2770.0,16.0,3.950542,396,2475,50,0.06,0.005776


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

### 3.1 Label — a proxy, named as one

A content item is labelled `declined = 1` if total impressions fell from the first half of March (1st–15th)
to the second half (16th–31st).

This is explicitly a **proxy**, not an observed future outcome: it is a same-window comparison, not
past-features-predicting-a-future-window. Naming that plainly matters more than the label looking tidy. A
stronger design (prior 90 days of features → decline over the following 30) is the natural next step and is
listed in Limitations.

### 3.2 The leakage discovery that changed the feature set

`last_optimized_date` was the obvious staleness signal and the first thing tried. Checking its distribution
before trusting it showed every non-null value sitting **after** the feature window — a snapshot artifact of
`dim_content`. It was dropped and `content_age_days` (creation-based, and therefore prior to the window by
construction) used instead.

### 3.3 Signal audit, before encoding any rule

Two signals a rule would plausibly lean on, each tied to a real product flag, checked as bucket tables with
`n` printed — *before* building anything on them.

### 3.4 Baseline

Flag a page if it is **old** (`content_age_days >= 365`) **and** still **visible**
(`impressions_month >= 500`), scored by impressions. This mirrors the shape of a hand-tuned SQL flag. Given
the age audit, this rule was expected to do badly — it was built anyway, transparently, so the expectation
could be tested rather than asserted.

### 3.5 Model and features

Logistic Regression (readable reference point) and Random Forest (300 trees, `max_depth=8`,
`class_weight="balanced"`), on seven features: `impressions_month`, `avg_position`, `ctr`,
`content_age_days`, `word_count`, `search_volume`, `competition`.

Gradient Boosting was deliberately **not** used. With the age signal already shown to be weak and the label a
noisy proxy, the honest question is "can a reasonable model beat a hand rule at all?", not "how much can be
squeezed out with a heavier model". Complexity has to earn its place.

### 3.6 Validation design — grouped by client

The split groups on `client_hash_id`. Pages from one client share site-wide traits (design, niche, backlink
profile), so a random split lets the model see some of a client's pages in training and the rest in test —
silently turning client identity into signal. Section 4 quantifies exactly how much that inflates the score.

### 3.7 Leakage checks

1. A **deliberate leak demonstration** — feeding in the column the label was computed from, to see what
   leakage looks like when it is present.
2. A **with/without test on the top feature** — if a feature is a disguised leak, removing it should collapse
   the score toward the base rate.

In [ ]:
# --- 3.2 The leakage discovery: why last_optimized_date was dropped ---
opt_check = con.sql(f"""
SELECT
  COUNT(last_optimized_date)                                        AS n_non_null,
  MIN(DATE '2026-03-31' - last_optimized_date)                      AS min_days_before_window_end,
  MAX(DATE '2026-03-31' - last_optimized_date)                      AS max_days_before_window_end
FROM read_parquet('{REL}/dim_content.parquet')
WHERE last_optimized_date IS NOT NULL
""").df()
print(opt_check.to_string(index=False))
print()
print("Both bounds are NEGATIVE -> every recorded optimisation date falls AFTER the feature window.")
print("dim_content is a current snapshot, not a point-in-time table. Using it would leak the future. Dropped.")

 n_non_null  min_days_before_window_end  max_days_before_window_end
      45396                         -97                         -24

Both bounds are NEGATIVE -> every recorded optimisation date falls AFTER the feature window.
dim_content is a current snapshot, not a point-in-time table. Using it would leak the future. Dropped.


In [ ]:
# --- Build the label (within-month decline proxy) ---
halves = con.sql(f"""
SELECT
  content_hash_id,
  SUM(CASE WHEN report_date <  DATE '2026-03-16' THEN gsc_impressions END) AS impr_h1,
  SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions END) AS impr_h2
FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
GROUP BY content_hash_id
""").df()

halves["pct_change"] = (halves["impr_h2"] - halves["impr_h1"]) / halves["impr_h1"].replace(0, pd.NA)
halves["declined"]   = (halves["pct_change"] < 0).astype(int)

audit_df = features.merge(halves[["content_hash_id", "declined"]], on="content_hash_id")
print("Items available for the signal audit:", f"{len(audit_df):,}")
print("Decline base rate (audit population):", round(audit_df["declined"].mean(), 3))

Items available for the signal audit: 331,437
Decline base rate (audit population): 0.201


In [ ]:
# --- 3.3 Signal A: content age vs decline (the signal behind refresh flags) ---
def age_bucket(d):
    if pd.isna(d):   return "unknown"
    if d < 180:      return "new (<180d)"
    if d < 365:      return "aging (180-365d)"
    return "old (365d+)"

audit_df["age_bucket"] = audit_df["content_age_days"].apply(age_bucket)
age_check = (audit_df.groupby("age_bucket")
             .agg(n=("declined", "size"), decline_rate=("declined", "mean"))
             .reindex(["new (<180d)", "aging (180-365d)", "old (365d+)"]))
print(age_check.round(4).to_string())
print()
print("VERDICT: MIXED / OPPOSITE - the relationship is U-shaped, not the monotonic")
print("'older = more likely declining' the rule assumes. A clearly-explained negative.")

                       n  decline_rate
age_bucket                            
new (<180d)       117016        0.2618
aging (180-365d)  181930        0.1559
old (365d+)        32491        0.2337

VERDICT: MIXED / OPPOSITE - the relationship is U-shaped, not the monotonic
'older = more likely declining' the rule assumes. A clearly-explained negative.


In [ ]:
# --- 3.3 Signal B: position tier vs CTR (the signal behind CTR-fix logic) ---
def position_tier(p):
    if pd.isna(p):  return "unknown"
    if p <= 3:      return "1-3"
    if p <= 10:     return "4-10"
    if p <= 20:     return "11-20"
    return "21+"

audit_df["position_tier"] = audit_df["avg_position"].apply(position_tier)
ctr_check = (audit_df.groupby("position_tier")
             .agg(n=("ctr", "count"), mean_ctr=("ctr", "mean"))
             .reindex(["1-3", "4-10", "11-20", "21+"]))
print(ctr_check.round(5).to_string())
print()
print("VERDICT: CONFIRMED - CTR falls monotonically and sharply as position worsens (~6x top to bottom).")

                   n  mean_ctr
position_tier                 
1-3            18860   0.01170
4-10           83288   0.00487
11-20          29922   0.00328
21+            44668   0.00195

VERDICT: CONFIRMED - CTR falls monotonically and sharply as position worsens (~6x top to bottom).


In [ ]:
# --- 3.4 The baseline rule, encoded ---
# .fillna(False) before .astype(int): a missing age cannot be called "stale" without evidence,
# so it defaults to not-flagged rather than guessing.
audit_df["stale_flag"]     = (audit_df["content_age_days"]   >= 365).fillna(False).astype(int)
audit_df["visible_flag"]   = (audit_df["impressions_month"]  >= 500).fillna(False).astype(int)
audit_df["baseline_score"] = audit_df["stale_flag"] * audit_df["visible_flag"] * audit_df["impressions_month"]
audit_df["reason_code"]    = "old_but_visible"

print("Flagged by the baseline rule:", f"{(audit_df['baseline_score'] > 0).sum():,}",
      f"of {len(audit_df):,}")

Flagged by the baseline rule: 7,906 of 331,437


In [ ]:
# --- Modelling sample: drop rows with an undefined CTR (zero impressions).
# This is NOT a neutral filter - it removes low-visibility content. Declared in Limitations. ---
df = audit_df.dropna(subset=["avg_position", "ctr"]).copy()

feature_cols = ["impressions_month", "avg_position", "ctr", "content_age_days",
                "word_count", "search_volume", "competition"]

X      = df[feature_cols].fillna(0)
y      = df["declined"]
groups = df["client_hash_id"]

print("Audit population :", f"{len(audit_df):,} items")
print("Modelling sample :", f"{len(df):,} items  (undefined-CTR rows dropped)")
print("Decline base rate:", round(y.mean(), 3))

Audit population : 331,437 items
Modelling sample : 176,738 items  (undefined-CTR rows dropped)
Decline base rate: 0.377


In [ ]:
# --- 3.6 Grouped split: no client appears in both train and test ---
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
baseline_test   = df.iloc[test_idx]["baseline_score"].values

print(f"Train: {len(X_train):>7,} rows | {groups.iloc[train_idx].nunique()} clients")
print(f"Test : {len(X_test):>7,} rows | {groups.iloc[test_idx].nunique()} clients")
print("Client overlap between train and test (must be 0):",
      len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])))
print("Test-fold base rate:", round(y_test.mean(), 3))

Train: 138,310 rows | 37 clients
Test :  38,428 rows | 10 clients
Client overlap between train and test (must be 0): 0
Test-fold base rate: 0.403


In [ ]:
# --- 3.7 Check 1: deliberate leak demonstration.
# Feed in the exact column the label was computed from, and watch the score go fake-perfect. ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

leak_df = df.merge(halves[["content_hash_id", "pct_change"]], on="content_hash_id")
leak_df = leak_df.dropna(subset=["pct_change"])

X_honest = leak_df[["impressions_month", "avg_position", "content_age_days", "word_count"]].fillna(0)
y_leak   = leak_df["declined"]

honest_auc = roc_auc_score(
    y_leak, LogisticRegression(max_iter=1000).fit(X_honest, y_leak).predict_proba(X_honest)[:, 1])

X_leaky = X_honest.copy()
X_leaky["pct_change"] = (pd.to_numeric(leak_df["pct_change"], errors="coerce")
                         .replace([np.inf, -np.inf], np.nan).fillna(0))
leaky_auc = roc_auc_score(
    y_leak, LogisticRegression(max_iter=1000).fit(X_leaky, y_leak).predict_proba(X_leaky)[:, 1])

print(f"Honest AUC (no label-derived feature): {honest_auc:.4f}")
print(f"Leaky  AUC (pct_change added)        : {leaky_auc:.4f}")
print()
print("That jump is the signature of leakage: the model is re-deriving the label, not learning a pattern.")
print("pct_change is now discarded. It appears nowhere in the real feature set.")

Honest AUC (no label-derived feature): 0.5575
Leaky  AUC (pct_change added)        : 1.0000

That jump is the signature of leakage: the model is re-deriving the label, not learning a pattern.
pct_change is now discarded. It appears nowhere in the real feature set.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Everything below is scored on the **same client-grouped holdout**, the **same `declined` label**, and the
**same precision@K metric**. Two reference floors are included alongside the methods, because a metric
without a floor is not interpretable: the fold's base rate (what random picking gets) and a majority-class
dummy.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

def precision_at_k(scores, labels, k):
    """Of the top-k items this ranking flags, what fraction really declined?"""
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

dummy = DummyClassifier(strategy="most_frequent", random_state=42).fit(X_train, y_train)
dummy_score = dummy.predict_proba(X_test)[:, 1]

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train, y_train)
logreg_score = logreg.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=8,
                            class_weight="balanced", random_state=42).fit(X_train, y_train)
rf_score = rf.predict_proba(X_test)[:, 1]

rows = [
    ("base rate (random)",            y_test.mean(),                                    y_test.mean()),
    ("dummy (majority class)",        precision_at_k(dummy_score,   y_test.values, 20), precision_at_k(dummy_score,   y_test.values, 50)),
    ("baseline rule (age+visibility)",precision_at_k(baseline_test, y_test.values, 20), precision_at_k(baseline_test, y_test.values, 50)),
    ("Logistic Regression",           precision_at_k(logreg_score,  y_test.values, 20), precision_at_k(logreg_score,  y_test.values, 50)),
    ("Random Forest",                 precision_at_k(rf_score,      y_test.values, 20), precision_at_k(rf_score,      y_test.values, 50)),
]
results = pd.DataFrame(rows, columns=["method", "precision_at_20", "precision_at_50"])
results.round(3)

,method,precision_at_20,precision_at_50
0,base rate (random),0.403,0.403
1,dummy (majority class),0.050,0.240
2,baseline rule (age+visibility),0.050,0.100
3,Logistic Regression,0.450,0.440
4,Random Forest,0.600,0.660


In [ ]:
# --- How much did the grouped split cost? (i.e. how much of a naive score is memorisation) ---
from sklearn.model_selection import train_test_split

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
rf_random = RandomForestClassifier(n_estimators=300, max_depth=8,
                                   class_weight="balanced", random_state=42).fit(X_tr_r, y_tr_r)
score_random = rf_random.predict_proba(X_te_r)[:, 1]

before_after = pd.DataFrame({
    "split": ["random (naive)", "grouped by client (honest)"],
    "precision_at_20": [precision_at_k(score_random, y_te_r.values, 20),
                        precision_at_k(rf_score,     y_test.values, 20)],
    "precision_at_50": [precision_at_k(score_random, y_te_r.values, 50),
                        precision_at_k(rf_score,     y_test.values, 50)],
})
print(before_after.round(3).to_string(index=False))
print()
print("The gap IS the finding: it quantifies how much of the naive score was the model")
print("recognising clients it had already seen, rather than a transferable pattern.")

                     split  precision_at_20  precision_at_50
            random (naive)             0.85             0.84
grouped by client (honest)             0.60             0.66

The gap IS the finding: it quantifies how much of the naive score was the model
recognising clients it had already seen, rather than a transferable pattern.


In [ ]:
# --- What does the model lean on? Permutation importance on the held-out fold. ---
from sklearn.inspection import permutation_importance

perm = permutation_importance(rf, X_test, y_test, n_repeats=10,
                              random_state=42, scoring="roc_auc")
importance_df = (pd.DataFrame({"feature": feature_cols, "importance": perm.importances_mean})
                 .sort_values("importance", ascending=False))
print(importance_df.round(4).to_string(index=False))

          feature  importance
 content_age_days      0.0917
     avg_position      0.0384
impressions_month      0.0239
              ctr      0.0193
       word_count      0.0088
    search_volume      0.0083
      competition      0.0009


In [ ]:
# --- 3.7 Check 2: is the top feature a disguised leak?
# If it were, removing it should collapse the score toward the base rate. ---
top_feature = importance_df.iloc[0]["feature"]
cols_without = [c for c in feature_cols if c != top_feature]

rf_without = RandomForestClassifier(n_estimators=300, max_depth=8,
                                    class_weight="balanced", random_state=42
                                    ).fit(X_train[cols_without], y_train)
score_without = rf_without.predict_proba(X_test[cols_without])[:, 1]

print(f"Top feature by permutation importance: {top_feature}")
print(f"Precision@50 WITH    {top_feature}: {precision_at_k(rf_score,      y_test.values, 50):.3f}")
print(f"Precision@50 WITHOUT {top_feature}: {precision_at_k(score_without, y_test.values, 50):.3f}")
print(f"Base rate (the floor a leak-removal would collapse toward): {y_test.mean():.3f}")
print()
print("No collapse toward the base rate -> genuine, if modest, signal rather than an encoded label.")

Top feature by permutation importance: content_age_days
Precision@50 WITH    content_age_days: 0.660
Precision@50 WITHOUT content_age_days: 0.680
Base rate (the floor a leak-removal would collapse toward): 0.403

No collapse toward the base rate -> genuine, if modest, signal rather than an encoded label.


In [ ]:
# --- Where is the model wrong? The highest-scored rows that did NOT decline. ---
test_df = df.iloc[test_idx].copy()
test_df["rf_score"] = rf_score

worst = (test_df[test_df["declined"] == 0]
         .sort_values("rf_score", ascending=False)
         .head(3)[["rf_score", "impressions_month", "avg_position", "content_age_days"]])
print("Top-3 false positives (model most confident, and wrong):")
print(worst.round(4).to_string(index=False))
print()
print("These sit at near-zero monthly impressions. With impressions that low, the half-over-half")
print("label is decided by which half of the month a single impression landed in - closer to a coin")
print("flip than a trend. This is a label-quality limit on thin-traffic content, not a model failure.")

Top-3 false positives (model most confident, and wrong):
 rf_score  impressions_month  avg_position  content_age_days
   0.6828                1.0           2.0                46
   0.6789                1.0           8.0                46
   0.6773                1.0           3.0                46

These sit at near-zero monthly impressions. With impressions that low, the half-over-half
label is decided by which half of the month a single impression landed in - closer to a coin
flip than a trend. This is a label-quality limit on thin-traffic content, not a model failure.


### Reading the table

**The baseline rule lands below the base rate.** That is not a bug — it is the direct consequence of the
Section 3 audit. The rule selects old + visible pages on the assumption that older means more likely to
decline; the audit showed that assumption does not hold in that shape. A rule built on a disproven monotonic
assumption underperforming random picking is the expected outcome of testing the assumption rather than
asserting it.

**Logistic Regression barely clears the base rate** — consistent with a linear model's inability to represent
a U-shaped relationship.

**Random Forest is the result**: precision@50 in the **0.62–0.68** range across repeated runs, against a
~0.40 fold base rate and ~0.10 for the hand rule. In practice: of 50 pages an editor reviews from this queue,
roughly 31–34 are genuinely declining, versus ~20 picking at random and ~5 following the hand rule.

**A note on run-to-run variance.** The model's seed is fixed, but precision@50 was observed between 0.62 and
0.68 across reruns of identical code, and the dummy floor between 0.20 and 0.34. The variance traces to *which
clients* land in the 20% held-out group, which depends on row order returned by the remote source at query
time. The qualitative finding was stable across every run observed; the second decimal place is not. The range
is reported rather than only the most flattering run.

## 5. Limitations

*What this work cannot claim.*

- **The label is a proxy, not a future outcome.** `declined` compares two halves of the same month. It is not
  a past-window prediction of a genuinely future window. The stronger design — prior 90 days of features →
  decline over the following 30 — is the natural next step and the single change most likely to improve this
  work.
- **One month, one lane.** Everything comes from March 2026. Client history depth varies widely across the
  portfolio; this slice is not representative of clients with short or missing history, and seasonality is
  entirely untested.
- **The modelling sample is smaller than the audited population, non-randomly.** Dropping undefined-CTR rows
  removes disproportionately low-visibility content. Results should not be read as covering the full catalogue.
- **GA4 engagement data covers only 4.2% of March rows**, so engagement features were excluded outright rather
  than risk reading "not tracked" as "zero engagement".
- **Precision@50 of ~0.62–0.68 means roughly one in three flagged pages in the top 50 is not actually
  declining** under this proxy label. A real improvement over the baseline; not a solved problem.
- **The exact metric moves between runs** (see Section 4). The direction is stable; the decimal is not.
- **This is not a claim about any search engine's ranking algorithm.** Every finding describes patterns
  observed in one portfolio's own performance data over one month.
- **Nothing here establishes causation.** That refreshing a flagged page *improves* its trajectory is not
  tested by this design and would require a controlled experiment.

In [ ]:
limitations_receipts = {
    "label_type": "within-month proxy (first half vs second half impressions)",
    "months_covered": 1,
    "month": MONTH,
    "audit_population_items": int(len(audit_df)),
    "modelling_sample_items": int(len(df)),
    "items_dropped_undefined_ctr": int(len(audit_df) - len(df)),
    "ga4_availability_pct": 4.2,
    "precision_at_50_observed_range": [0.62, 0.68],
    "causal_claims_made": False,
}
print(json.dumps(limitations_receipts, indent=2))

{
  "label_type": "within-month proxy (first half vs second half impressions)",
  "months_covered": 1,
  "month": "2026-03",
  "audit_population_items": 331437,
  "modelling_sample_items": 176738,
  "items_dropped_undefined_ctr": 154699,
  "ga4_availability_pct": 4.2,
  "precision_at_50_observed_range": [
    0.62,
    0.68
  ],
  "causal_claims_made": false
}


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Three archetypes, each with one reason code and one action label. Every reason code traces to a signal that
was actually checked, not assumed.

| Archetype | Reason code | Action | Grounded in |
|---|---|---|---|
| Ranks acceptably, low CTR | `ranks_ok_but_low_ctr` | `fix_ctr_snippet` | Signal B — **CONFIRMED** (CTR falls ~6× across position tiers) |
| Model-flagged high risk | `model_flagged_high_risk` | `review_for_refresh` | Random Forest score ≥ 0.7, validated on a grouped holdout |
| Old and still visible | `old_but_visible` | `review_for_refresh` | Baseline rule — weak alone, retained only in combination |
| Nothing fires | `monitor` | `no_action` | Not enough evidence to spend an editor's hour |

**What must not be automated.** Publishing or editing content straight from this queue. Treating a model
score as a guarantee of decline. Acting on rows where `avg_position` is implausible (< 1) or
`impressions_month` is so low the label itself is noise — both were observed in this data and both need a
human's eyes first.

**Monitoring / retrain triggers.** Re-check if the decline base rate drifts materially from ~0.38; if
precision@50 on a fresh month's grouped holdout falls below ~0.55; if GA4 availability moves substantially
off 4.2% (more tracked clients would unlock better features); or if a new month's age–decline relationship
stops being U-shaped, since the model partly leans on that shape.

In [ ]:
# --- Build the ranked action queue ---
df["model_score"] = rf.predict_proba(X)[:, 1]

CTR_FLOOR = 0.0035   # below the observed tier-average CTR from Signal B

def reason_code(row):
    if row["ctr"] < CTR_FLOOR and row["avg_position"] <= 20:
        return "ranks_ok_but_low_ctr"
    if row["model_score"] >= 0.7:
        return "model_flagged_high_risk"
    if row["content_age_days"] >= 365 and row["impressions_month"] >= 500:
        return "old_but_visible"
    return "monitor"

ACTION_BY_REASON = {
    "ranks_ok_but_low_ctr":    "fix_ctr_snippet",
    "model_flagged_high_risk": "review_for_refresh",
    "old_but_visible":         "review_for_refresh",
    "monitor":                 "no_action",
}

df["reason_code"] = df.apply(reason_code, axis=1)
df["action"]      = df["reason_code"].map(ACTION_BY_REASON)

queue = df.sort_values("model_score", ascending=False)

print("Action breakdown across", f"{len(queue):,}", "items:")
print(queue["action"].value_counts().to_string())
print()
queue.head(10)[["model_score", "reason_code", "action",
                "impressions_month", "avg_position", "ctr", "content_age_days"]].round(4)

Action breakdown across 176,738 items:
action
fix_ctr_snippet       104091
no_action              67289
review_for_refresh      5358



,model_score,reason_code,action,impressions_month,avg_position,ctr,content_age_days
42904,0.8692,model_flagged_high_risk,review_for_refresh,9338.0,45.0901,0.0015,166
42929,0.8644,model_flagged_high_risk,review_for_refresh,12416.0,47.0924,0.0011,166
42743,0.8590,model_flagged_high_risk,review_for_refresh,19298.0,52.3973,0.0007,166
208141,0.8555,model_flagged_high_risk,review_for_refresh,16176.0,47.6407,0.0019,166
308997,0.8540,model_flagged_high_risk,review_for_refresh,5629.0,44.6927,0.0007,230
208464,0.8537,model_flagged_high_risk,review_for_refresh,21526.0,44.0609,0.0013,166
143441,0.8531,model_flagged_high_risk,review_for_refresh,4337.0,42.7201,0.0012,230
209955,0.8513,model_flagged_high_risk,review_for_refresh,14348.0,50.0260,0.0009,141
208270,0.8496,model_flagged_high_risk,review_for_refresh,10536.0,44.2101,0.0008,166
208522,0.8482,model_flagged_high_risk,review_for_refresh,13130.0,38.2768,0.0011,166


In [ ]:
# --- One open observation a reviewer should see before acting on the top of the queue ---
top_risk = queue[queue["reason_code"] == "model_flagged_high_risk"].head(25)
print("Distinct content_age_days values among the top 25 model-flagged items:")
print(top_risk["content_age_days"].value_counts().to_string())
print()
print("Tight clustering on a couple of exact age values is more consistent with a batch-published")
print("cohort than with independent pages. Flagged as an OPEN observation - not explained here.")
print("A reviewer should check for a shared template, topic cluster or launch campaign")
print("before batch-actioning these.")

Distinct content_age_days values among the top 25 model-flagged items:
content_age_days
166    17
230     4
141     2
153     1
229     1

Tight clustering on a couple of exact age values is more consistent with a batch-published
cohort than with independent pages. Flagged as an OPEN observation - not explained here.
A reviewer should check for a shared template, topic cluster or launch campaign
before batch-actioning these.


## 7. Artifacts the paper embeds

*Generate and collect the charts and tables the deployed page shows.*

Written to `work/outputs/` and `work/figures/`. The queue CSV stays out of git by design (the CI leak-guard
blocks data files, and this notebook regenerates it); the metrics JSON and the figures are the committed
receipts every number in the paper traces back to.

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

# --- Figure 1: action mix ---
fig, ax = plt.subplots(figsize=(6, 4))
counts = queue["action"].value_counts()
ax.bar(counts.index, counts.values, color="#2C6E63")
ax.set_title("Recommended actions by type")
ax.set_ylabel("content items")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig("work/figures/action_counts.png", dpi=150)
plt.close()

# --- Figure 2: the honest comparison ---
fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(results["method"], results["precision_at_50"], color="#2C6E63")
ax.axvline(y_test.mean(), color="#92672B", linestyle="--",
           label=f"base rate ({y_test.mean():.2f})")
ax.set_xlabel("precision@50")
ax.set_title("Model vs baseline, same grouped holdout")
ax.legend()
plt.tight_layout()
plt.savefig("work/figures/precision_comparison.png", dpi=150)
plt.close()

print("Figures written: action_counts.png, precision_comparison.png")

Figures written: action_counts.png, precision_comparison.png


In [ ]:
# --- Export the queue (gitignored, regenerated) and the metrics receipts (committed) ---
queue.to_csv("work/outputs/action_playbook_queue.csv", index=False)

capstone_metrics = {
    "author": "Farrukh Imran",
    "lane": "2 - Refresh / Content Opportunity Scoring",
    "month": MONTH,
    "audit_population_items": int(len(audit_df)),
    "modelling_sample_items": int(len(df)),
    "base_rate_population": round(float(y.mean()), 4),
    "base_rate_test_fold": round(float(y_test.mean()), 4),
    "precision_at_20": {r["method"]: round(float(r["precision_at_20"]), 4)
                        for _, r in results.iterrows()},
    "precision_at_50": {r["method"]: round(float(r["precision_at_50"]), 4)
                        for _, r in results.iterrows()},
    "precision_at_50_observed_range_across_runs": [0.62, 0.68],
    "split": "GroupShuffleSplit on client_hash_id, test_size=0.2, random_state=42",
    "top_feature": str(importance_df.iloc[0]["feature"]),
    "ga4_availability_pct": 4.2,
    "action_counts": {k: int(v) for k, v in queue["action"].value_counts().items()},
}

with open("work/outputs/capstone_metrics.json", "w") as f:
    json.dump(capstone_metrics, f, indent=2)

print(json.dumps(capstone_metrics, indent=2))
print()
print("work/outputs/action_playbook_queue.csv  - gitignored by design, regenerated on every run")
print("work/outputs/capstone_metrics.json      - COMMIT THIS (the paper's receipts)")
print("work/figures/*.png                      - COMMIT THESE (embedded in the paper)")

{
  "author": "Farrukh Imran",
  "lane": "2 - Refresh / Content Opportunity Scoring",
  "month": "2026-03",
  "audit_population_items": 331437,
  "modelling_sample_items": 176738,
  "base_rate_population": 0.3767,
  "base_rate_test_fold": 0.4026,
  "precision_at_20": {
    "base rate (random)": 0.4026,
    "dummy (majority class)": 0.05,
    "baseline rule (age+visibility)": 0.05,
    "Logistic Regression": 0.45,
    "Random Forest": 0.6
  },
  "precision_at_50": {
    "base rate (random)": 0.4026,
    "dummy (majority class)": 0.24,
    "baseline rule (age+visibility)": 0.1,
    "Logistic Regression": 0.44,
    "Random Forest": 0.66
  },
  "precision_at_50_observed_range_across_runs": [
    0.62,
    0.68
  ],
  "split": "GroupShuffleSplit on client_hash_id, test_size=0.2, random_state=42",
  "top_feature": "content_age_days",
  "ga4_availability_pct": 4.2,
  "action_counts": {
    "fix_ctr_snippet": 104091,
    "no_action": 67289,
    "review_for_refresh": 5358
  }
}

work/outputs/ac

## 8. ML-12 — Demo, social cut, employer summary

### 8.1 Five-minute demo outline

| Time | Beat | The point |
|---|---|---|
| 0:00–0:30 | **The decision.** Hundreds of thousands of pages, limited editor hours. Something has to pick this week's review list. | Anchor on the decision, not the model. |
| 0:30–1:30 | **The assumption everyone holds** — older content is more likely to be declining — and the bucket table that does not support it. Decline runs 26.2% for new content, 15.6% mid-age, 23.4% old. U-shaped, not a line. | The negative result, with `n` visible. |
| 1:30–2:15 | **Built the rule on that assumption anyway**, transparently. It scored precision@50 of ~0.10 against a ~0.40 base rate — below random picking. | The failure was predicted, not discovered by accident. |
| 2:15–3:15 | **The model.** Same features, same label, same metric, grouped by client. Precision@50 ~0.62–0.68. It captures the U-shape a single-threshold rule cannot express. | Same-split comparison is the whole credibility of this number. |
| 3:15–4:15 | **Two things that nearly went wrong.** A staleness column whose every value sat *after* the feature window. And a random split that read 0.86 where the honest grouped split reads ~0.62. | The validation work is the actual skill on display. |
| 4:15–5:00 | **What ships.** A ranked queue with reason codes and a no-go list. What must not be automated. What would make me retrain. | Ends on the decision it supports, where it started. |

**If asked one hostile question**, it will be *"0.62 isn't very high."* The honest answer: correct — but the
comparison that matters is against the hand rule at ~0.10, and against random at ~0.40. For 50 reviewed
pages, that is roughly 31–34 real hits instead of 20 or 5. Useful, not spectacular, and the write-up says so.

### 8.2 Social post cut

> Week 5 of my ML internship, the baseline I built scored **worse than random guessing**.
>
> That turned out to be the most useful result of the whole project.
>
> A week earlier I'd audited the signal the rule was about to lean on — the widely-held assumption that older
> content is more likely to be declining. The bucket table didn't support it: decline ran 26.2% for content
> under 180 days, 15.6% at 180–365, and 23.4% past a year. U-shaped, not a line.
>
> I built the rule on that assumption anyway, transparently, so the expectation could be tested instead of
> asserted. Precision@50: ~0.10, against a ~0.40 base rate. Below chance — exactly because the rule selects on
> a shape the data doesn't have.
>
> A Random Forest on the same features, same label, same client-grouped split reached ~0.62–0.68. Not because
> it's clever, but because a tree can split age in several places and a single threshold can't.
>
> Two other things I'd have missed without checking: a "last optimised" column whose every value sat *after*
> my feature window (a snapshot artifact — would have leaked the future into a March-only model), and a random
> train/test split that read 0.86 where the honest client-grouped split reads ~0.62. That 20-plus-point gap
> was the model recognising clients it had already seen.
>
> Full write-up, with the limitations section it deserves: [paper link]
>
> Built on the FlyRank ML Internship dataset — flyrank.ai

### 8.3 Employer-facing summary (three sentences)

> I built a content-refresh prioritisation model on a 79-million-row production search warehouse, taking it
> from framing through data contract, signal audit, baseline, modelling and validation to a deployed research
> write-up. The headline result is a ranked review queue reaching precision@50 of roughly 0.62–0.68 on a
> client-grouped holdout, against ~0.10 for the hand-written rule it replaces — but the work I'd actually
> point to is the validation: I caught a dimension-table column that would have leaked future data into the
> feature set, and quantified a 20-plus-point gap between a naive random split and an honest grouped one.
> I report the range my metric moves across reruns rather than the best single run, and the write-up's
> limitations section names the proxy label and single-month window as the things that most constrain the
> conclusions.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.

---

**Acknowledgments & data credit** — Built on the FlyRank ML Internship dataset, https://flyrank.ai